# Test Notebook for baseline.py

This notebook tests the timbre transfer functions from `src/baseline.py`.
It uses a voice as the source and a violin as the target, similar to `playground.ipynb`.

In [ ]:
import numpy as np
import soundfile as sf
import pyworld as pw
import librosa
import matplotlib.pyplot as plt
import librosa.display
from IPython.display import Audio, display
from pathlib import Path
import sys

# Add src to path to import baseline
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features.baseline import Baseline
from src.analysis.visualize import plot_spectrograms

# Define paths
source_audio_path = PROJECT_ROOT / "data/VocalSet/FULL/female1/long_tones/forte/f1_long_forte_a.wav"
target_audio_path = PROJECT_ROOT / "data/IV. Double Presto.wav"
output_dir = PROJECT_ROOT / "data" / "baseline_outputs"
output_dir.mkdir(exist_ok=True)

# Display source and target audio
print("Source audio (voice):")
display(Audio(filename=source_audio_path))
print("Target audio (violin):")
display(Audio(filename=target_audio_path))

## 1. Load and Preprocess Audio
Load audio files and extract WORLD parameters (F0, spectral envelope, aperiodicity).

In [ ]:
# Load audio
baseline = Baseline(sample_rate=16000)
y_src, sr = baseline.load_audio(path = source_audio_path)
y_tgt, _ = baseline.load_audio(path = target_audio_path, sr=sr)

y_tgt = baseline.match_length(y_tgt, len(y_src))

# Decompose using WORLD
output1 = baseline.f0_transfer(y_src, y_tgt)
output2 = baseline.f0_and_ap_transfer(y_src, y_tgt)
display(Audio(output1, rate=sr))
display(Audio(output2, rate=sr))
# Plot spectrograms
plot_spectrograms([y_src, y_tgt, output1, output2  ], sr)

In [ ]:
f0_src,sp_src,ap_src = baseline.decompose(y_src,fs = sr)
f0_tgt,sp_tgt,ap_tgt = baseline.decompose(y_tgt,sr)

#fo transfer without ap

ap_zero = np.zeros_like(ap_src)

output3 = baseline.synthesize(f0_src,sp_tgt,ap_zero,sr)

display(Audio(output3, rate=sr))
# Plot spectrograms
plot_spectrograms([y_src, y_tgt, output3, output2  ], sr)

In [ ]:
f0_zeros = np.full_like
output4 = baseline.synthesize(f0_zeros,sp_tgt,ap_zero,sr)

display(Audio(output4, rate=sr))
# Plot spectrograms
plot_spectrograms([y_src, y_tgt, output4  ], sr)